# Wine quality classification — exploratory notebook

**Historical research notebook:** refactored file paths and cleared previous execution outputs. Run from the project subfolder (the folder containing `data/`). For a clean, reproducible holdout benchmark, run `evaluate.py` instead. Results in this notebook are experimental and are not the reported benchmark.


In [ ]:
# Внимание!!! Важно, что бы файлы с данными и исполняемый файл находились в одной папке, 
# тогда пути к тестовым и тренировочным наборам будут содержать только имена файлов.
# 
# В пути к тренировочным и тестовым данным запрежается использовать абсалютную адресацию, 
# то есть адресацию, в которой присутствуют имена папок. Путь должен содержать только имя файла.
#
# Напоминание: под моделью машинного обучения понимаются все действия с исходными данными, 
# которые необходимо произвести, что бы сопоставить признаки целевому значению.

## Data exploration and setup

Библиотеки и функции

In [ ]:
# Данный блок в области 1 НЕ выполняется преподавателем
# 
# данный блок предназначен только для подключения необходимых библиотек
# запрещается подключать библиотеки в других блоках
# запрещается скрывать предупреждения системы
# установка дополнительных библиотек размещается прямо здесь (обязательно закоментированы)
# pip install

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, Normalizer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold 
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.metrics import recall_score, precision_score, f1_score, fbeta_score

from sklearn.linear_model import LogisticRegression

import sklearn

In [ ]:
def histmaker_for_numeric_df(df):
    """рисует хистплоты для каждого столбца датафрейма. 
    Если в дф-е есть столбцы, которые не нужно рисовать, их стоит удалить """
    #оставим только численные столбцы 
    cond = [np.issubdtype(df[f'{column}'], np.number) for column in df.columns]
    df = df[df.columns[cond]]
    
    sns.set(style='darkgrid', palette='deep')

    array_of_columns_of_df = df.values.T.tolist()

    num_plots = len(df.columns)         #количество графиков
    num_rows = (num_plots - 1) // 3 + 1 #количество строк 
    num_cols = min(num_plots, 3)        #количество столбцов

    fig, axes = plt.subplots(num_rows, num_cols, figsize = (15, 4*num_rows))
    plt.suptitle('Histograms of every column in your df', fontsize = 15)
    for i, ax in enumerate(axes.flat):
        col = (np.random.random (), np.random.random (), np.random.random ()) #случайный увет
        if i < num_plots:
            ax.hist(array_of_columns_of_df[i],
                    bins = 20,
                    edgecolor = 'black', color = col, alpha = 0.7)
            ax.set_title(df.columns[i], weight='bold')
        if i % 3 == 0:
            ax.set_ylabel('count')
    
    #Удаление пустых подграфиков
    for i in range(num_plots, num_rows * num_cols):
        fig.delaxes(axes.flatten()[i])


## Model search and evaluation

Поиск лучшей модели

**Загрузим данные и посмотрим на них**

In [ ]:
wine = pd.read_csv('data/train.csv')
wine

In [ ]:
wine.describe()

In [ ]:
histmaker_for_numeric_df(wine)

In [ ]:
dataplot = sns.heatmap(wine.corr(), annot = True)

In [ ]:
wine = wine.drop(columns = ['density'])

In [ ]:
dataplot = sns.heatmap(wine.corr(), annot = True,vmin=-1, vmax=1)

In [ ]:
wine['quality'] = np.where(wine.quality >= 7, 1, 0)
wine

In [ ]:
# доля классов в изначальных данных
wine['quality'].mean()
#количество классов неравномерное

**Разделим выборку на тест и трейн**

In [ ]:
wine_target = np.array(wine['quality'].values)
wine_date = wine.drop(columns = ['quality'])




In [ ]:
X_train, X_test, y_train, y_test = train_test_split(wine_date, wine_target, stratify = wine_target, test_size=0.3)


**Подберем число соседей**

In [ ]:
# Объявление контейнера
pipe = Pipeline([('preprocessing', RobustScaler()), 
                 ('classifier',    KNeighborsClassifier())])

# Объявление кросс-валидации
kfold = KFold(n_splits=10, shuffle=True)

# Объявление параметров гридсерча
p = np.arange(1,4)
n_neighbors = np.arange(90, 150)
scaling = [MinMaxScaler(), StandardScaler(),RobustScaler()]

# Засунем параметры гридсерча в него
param_grid =[
    {'preprocessing': scaling, 
     'classifier': [KNeighborsClassifier()],
         'classifier__n_neighbors': n_neighbors,
         'classifier__p': p,
         'classifier__weights': ['uniform','distance']
     }]

grid = GridSearchCV(pipe, param_grid, cv=kfold, scoring = 'roc_auc')
grid.fit(X_train,y_train)
grid

In [ ]:
#sklearn.metrics.get_scorer_names()

In [ ]:
# тестовый скор
grid.best_score_


In [ ]:
#тренировочный скор
grid.score(X_test, y_test)

In [ ]:
grid_result = pd.DataFrame(grid.cv_results_).sort_values(["rank_test_score",'std_test_score']).T
grid_result

In [ ]:
mask = grid_result.index[grid_result.index.str.endswith("_test_score")][:-3]
tabl = grid_result.loc[mask].iloc[:,:10]
sns.boxplot(data=tabl);

In [ ]:
grid.best_params_

In [ ]:
pipe = Pipeline([('preprocessing', MinMaxScaler()), 
                 ('classifier',    KNeighborsClassifier(n_neighbors=106,
                                                        p = 1,
                                                        weights='distance'))])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

#это метрики при данном количестве соседей, но порог принятия решения не оптимизирован
print(f'precision:     {precision_score(y_test, y_pred)}')
print(f'recall:        {recall_score(y_test, y_pred)}')
print(f'f1:            {f1_score(y_test, y_pred)}')
print(f'roc_auc:       {roc_auc_score(y_test, y_pred)} \n')
print(classification_report(y_test, y_pred))

In [ ]:
best_score = -1
threshold_best = -1
y_pred_proba = pipe.predict_proba(X_test)[:,1]

#попытаемся подвигать порог в сторону увеличения пресижн
for threshold in np.unique(pipe.predict_proba(X_test)[:,1]):
        y_pred = (y_pred_proba > threshold).astype(int)
        scr = fbeta_score(y_test, y_pred, beta = 0.1)
        if scr > best_score:
            best_score = scr
            threshold_best = threshold
print(f'Treshold: {threshold_best}') # этот порог берем в модель

In [ ]:
y_pred = (y_pred_proba > threshold_best).astype(int)

print('precision', precision_score(y_test, y_pred))
print('recall', recall_score(y_test, y_pred))
print('fbeta', fbeta_score(y_test, y_pred, beta = 0.4))
print('ROC AUC', roc_auc_score(y_test, y_pred))

#финальные метрики

### Попробуем logreg

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.datasets import load_iris

from sklearn.pipeline import make_pipeline, Pipeline

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.preprocessing import RobustScaler, Normalizer

from sklearn.model_selection import train_test_split, cross_val_score  
from sklearn.model_selection import GridSearchCV, KFold, StratifiedShuffleSplit

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report

import sklearn
print(sklearn.__version__)

In [ ]:
pipe = Pipeline([('preprocessing', MinMaxScaler()), 
                 ('clf',           LogisticRegression())])

cv = StratifiedShuffleSplit(test_size = .3, train_size = .6, n_splits = 10)
scaling = [ MinMaxScaler(), StandardScaler(),RobustScaler(), Normalizer()]

param_grid =[
    {'preprocessing': scaling,'clf__penalty': ['l2'], 
         'clf__solver': ['newton-cg' ,'lbfgs', 'liblinear', 'sag', 'saga'],
          'clf__C' : [0.1, 0.3, 0.5, 0.7, 0.8, 1, 1.1, 1.3, 1.5]},
    {'preprocessing': scaling,'clf__penalty': ['l1'], 
         'clf__solver': ['liblinear','saga'],
          'clf__C' : [0.1, 0.3, 0.5, 0.7, 0.8, 1, 1.1, 1.3, 1.5]},
    {'preprocessing': scaling,'clf__penalty': ['none'], 
         'clf__solver': ['lbfgs','newton-cg','saga','sag'],
          'clf__C' : [0.1, 0.3, 0.5, 0.7, 0.8, 1, 1.1, 1.3, 1.5]},
    {'preprocessing': scaling,'clf__penalty': ['elasticnet'], 
         'clf__solver': ['saga'],
          'clf__C' : [0.1, 0.3, 0.5, 0.7, 0.8, 1, 1.1, 1.3, 1.5]}
    ]

grid = GridSearchCV(pipe, param_grid, cv = cv, scoring = 'roc_auc', return_train_score = True)
grid.fit(X_train, y_train)
grid    

In [ ]:
grid_result = pd.DataFrame(grid.cv_results_).sort_values(["rank_test_score",'std_test_score']).T
grid_result

In [ ]:
mask = grid_result.index[grid_result.index.str.endswith("_test_score")][:-3]
tabl = grid_result.loc[mask].iloc[:,:10]
sns.boxplot(data=tabl);

In [ ]:
grid.best_params_

In [ ]:
pipe_best = make_pipeline(RobustScaler(), LogisticRegression(solver = 'newton-cg', penalty = 'l2', C = 0.1) )

pipe_best.fit(X_train,y_train)

y_pred_best = pipe_best.predict(X_test)

print(f'precision:     {precision_score(y_test, y_pred_best)}')
print(f'recall:        {recall_score(y_test, y_pred_best)}')
print(f'f1:            {f1_score(y_test, y_pred_best)}')
print(f'roc_auc:       {roc_auc_score(y_test, y_pred_best)} \n')
print(classification_report(y_test, y_pred_best))

Судя по всему, модель из метода соседей лучше модели logreg. 

Из всех метрик логичнее всего мерить по precision, так как именно так мы минимизируем попадание плохих вин в нашу коллекцию хорошего вина (то есть минимизируем FP)